# 03b — PCA From Scratch

## The One-Sentence Version
PCA finds the eigenvectors of the covariance matrix — and we're going to build that from numpy before touching sklearn.

## What You'll Build Intuition For
- Why centering data is essential
- What the covariance matrix encodes
- How eigendecomposition extracts the principal directions
- How projection and reconstruction work

## Prerequisites
- [03a — Projection Intuition](03a_projection_intuition.ipynb) — what projection means geometrically

## The Story

In 03a we discovered that PCA is about finding the best angle to cast a shadow. The "best" angle maximises variance, which is the same as minimising reconstruction error.

Now we build the algorithm, piece by piece, using nothing but numpy. No sklearn, no black boxes. Each step has a clear purpose, and by the end you'll have a 20-line PCA implementation that matches sklearn exactly.

The steps are: centre the data, compute the covariance matrix, find its eigenvectors (the principal directions), project the data onto those directions, and optionally reconstruct. That's it. The whole of PCA.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from utils.plotting import apply_style, COLOURS, PALETTE
from utils.data_generators import make_low_d_in_high_d, make_digit_data

apply_style()
rng = np.random.default_rng(42)

## Step 1: Centre the Data

PCA needs mean-zero data. Why? Because PCA decomposes the **covariance** matrix, which measures how features vary *together around their means*. If the data isn't centred, the mean acts like a phantom signal that distorts the covariance.

In [ ]:
# Generate some 2D data with a non-zero mean
n = 300
angle = np.pi / 6  # 30 degrees
R = np.array([[np.cos(angle), -np.sin(angle)],
              [np.sin(angle),  np.cos(angle)]])
X_raw = rng.normal(0, 1, (n, 2)) @ np.diag([3.0, 0.8]) @ R.T
X_raw += np.array([5, 3])  # shift the mean

# Centre
mean = X_raw.mean(axis=0)
X_centered = X_raw - mean

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(X_raw[:, 0], X_raw[:, 1], s=10, alpha=0.5, color=COLOURS["signal"])
ax1.scatter(*mean, s=100, color=COLOURS["noise"], marker="x", linewidths=2,
            label=f"Mean ({mean[0]:.1f}, {mean[1]:.1f})")
ax1.set_xlabel("Feature 1")
ax1.set_ylabel("Feature 2")
ax1.set_title("Raw data (mean ≠ 0)")
ax1.legend()
ax1.set_aspect("equal")

ax2.scatter(X_centered[:, 0], X_centered[:, 1], s=10, alpha=0.5,
            color=COLOURS["signal"])
ax2.scatter(0, 0, s=100, color=COLOURS["noise"], marker="x", linewidths=2,
            label="Mean (0, 0)")
ax2.set_xlabel("Feature 1")
ax2.set_ylabel("Feature 2")
ax2.set_title("Centred data (mean = 0)")
ax2.legend()
ax2.set_aspect("equal")

fig.suptitle("Step 1: Centre the data", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03b_centering.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Before centering: mean = [{mean[0]:.2f}, {mean[1]:.2f}]")
print(f"After centering:  mean = [{X_centered.mean(0)[0]:.2e}, {X_centered.mean(0)[1]:.2e}]")

## Step 2: Compute the Covariance Matrix

The covariance matrix $C$ is a $d \times d$ matrix where entry $C_{ij}$ tells you how features $i$ and $j$ vary together.

$$C = \frac{1}{n-1} X^T X$$

- **Diagonal entries** = variances of each feature
- **Off-diagonal entries** = covariances between pairs

In [ ]:
# Compute covariance matrix manually, then verify with np.cov
C_manual = (X_centered.T @ X_centered) / (n - 1)
C_numpy = np.cov(X_centered.T)

print("Covariance matrix (manual):")
print(C_manual.round(4))
print(f"\nCovariance matrix (np.cov):")
print(C_numpy.round(4))
print(f"\nMatch: {np.allclose(C_manual, C_numpy)}")

print(f"\nDiagonal: variances = [{C_manual[0,0]:.2f}, {C_manual[1,1]:.2f}]")
print(f"Off-diagonal: covariance = {C_manual[0,1]:.2f}")
print(f"Correlation: r = {C_manual[0,1] / np.sqrt(C_manual[0,0] * C_manual[1,1]):.2f}")

## Step 3: Eigendecomposition

The eigenvectors of $C$ are the principal directions. The eigenvalues tell us how much variance each direction captures.

$$C \mathbf{v}_i = \lambda_i \mathbf{v}_i$$

The eigenvector with the largest $\lambda$ is PC1 — the direction of maximum variance.

In [ ]:
# Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(C_manual)

# Sort descending by eigenvalue
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("Eigenvalues (variance per component):")
for i, ev in enumerate(eigenvalues):
    print(f"  PC{i+1}: λ = {ev:.4f} ({ev/eigenvalues.sum():.1%} of total)")

print(f"\nEigenvectors (principal directions):")
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    print(f"  PC{i+1}: [{v[0]:.4f}, {v[1]:.4f}]")

# Visualise: eigenvectors as arrows on the data
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(X_centered[:, 0], X_centered[:, 1], s=10, alpha=0.4,
           color=COLOURS["signal"])

colours_pc = [COLOURS["noise"], COLOURS["accent"]]
for i in range(2):
    v = eigenvectors[:, i]
    scale = np.sqrt(eigenvalues[i]) * 2
    ax.annotate("", xy=v * scale, xytext=(0, 0),
                arrowprops=dict(arrowstyle="->", color=colours_pc[i], lw=2.5))
    ax.text(v[0]*scale*1.15, v[1]*scale*1.15, f"PC{i+1} (λ={eigenvalues[i]:.2f})",
            fontsize=11, fontweight="bold", color=colours_pc[i])

ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_title("Eigenvectors of the covariance matrix = principal components")
ax.set_aspect("equal")
ax.set_xlim(-8, 8)
ax.set_ylim(-8, 8)
plt.tight_layout()
plt.savefig("visuals/03b_eigenvectors.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 4: Project

Now we project the data onto the principal directions. This is just matrix multiplication:

$$X_{\text{pca}} = X_{\text{centered}} \cdot W$$

where $W$ is the matrix whose columns are the first $k$ eigenvectors.

In [ ]:
# Project onto first k=1 component
k = 1
W = eigenvectors[:, :k]  # shape (2, 1)
X_pca_1d = X_centered @ W  # shape (n, 1)

print(f"Original shape: {X_centered.shape}")
print(f"Projected shape: {X_pca_1d.shape}")
print(f"Reduced from 2D to {k}D\n")

# Verify against sklearn
pca_sklearn = PCA(n_components=1)
X_sklearn = pca_sklearn.fit_transform(X_raw)  # sklearn centers internally

# Flip sign if needed (eigenvector sign is arbitrary)
if np.corrcoef(X_pca_1d.ravel(), X_sklearn.ravel())[0, 1] < 0:
    X_pca_1d = -X_pca_1d

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(X_pca_1d.ravel(), bins=30, alpha=0.7, color=COLOURS["noise"],
         edgecolor="white", linewidth=0.3, label="Our PCA")
ax1.set_title("Our PCA (from scratch)")
ax1.set_xlabel("PC1 score")

ax2.hist(X_sklearn.ravel(), bins=30, alpha=0.7, color=COLOURS["signal"],
         edgecolor="white", linewidth=0.3, label="sklearn PCA")
ax2.set_title("sklearn PCA")
ax2.set_xlabel("PC1 score")

fig.suptitle("Our implementation matches sklearn", fontweight="bold")
plt.tight_layout()
plt.show()

max_diff = np.max(np.abs(X_pca_1d.ravel() - X_sklearn.ravel()))
print(f"Max difference between our PCA and sklearn: {max_diff:.2e}")
print("They're identical (up to floating point).")

## Step 5: Reconstruct

We can go back to the original space by reversing the projection:

$$X_{\text{reconstructed}} = X_{\text{pca}} \cdot W^T + \text{mean}$$

The reconstruction won't be perfect if we used fewer components than the data has dimensions — the difference is the reconstruction error.

In [ ]:
# Reconstruct from 1 component
X_reconstructed = X_pca_1d @ W.T + mean

# Reconstruction error
recon_error = np.mean(np.sum((X_raw - X_reconstructed)**2, axis=1))

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(X_raw[:, 0], X_raw[:, 1], s=10, alpha=0.3, color=COLOURS["signal"],
           label="Original")
ax.scatter(X_reconstructed[:, 0], X_reconstructed[:, 1], s=10, alpha=0.5,
           color=COLOURS["noise"], label="Reconstructed (1 PC)")

# Draw residual lines for a subset
for i in range(0, n, 10):
    ax.plot([X_raw[i, 0], X_reconstructed[i, 0]],
            [X_raw[i, 1], X_reconstructed[i, 1]],
            color=COLOURS["neutral"], alpha=0.3, linewidth=0.5)

ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.set_title(f"Reconstruction from 1 PC (error = {recon_error:.2f})")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("visuals/03b_reconstruction.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Reconstruction error from 1 PC: {recon_error:.3f}")
print(f"This equals the second eigenvalue: {eigenvalues[1]:.3f}")
print("The error IS the discarded variance.")

In [ ]:
# Reconstruction error vs number of components on higher-D data
X_high, Z_true = make_low_d_in_high_d(n=500, intrinsic_d=5, ambient_d=50, seed=42)
mean_high = X_high.mean(axis=0)
X_high_c = X_high - mean_high
C_high = np.cov(X_high_c.T)
evals_high, evecs_high = np.linalg.eigh(C_high)
idx_high = np.argsort(evals_high)[::-1]
evals_high = evals_high[idx_high]
evecs_high = evecs_high[:, idx_high]

errors = []
components_range = range(1, 21)
for k in components_range:
    W_k = evecs_high[:, :k]
    X_proj = X_high_c @ W_k
    X_recon = X_proj @ W_k.T
    errors.append(np.mean(np.sum((X_high_c - X_recon)**2, axis=1)))

explained = np.cumsum(evals_high[:20]) / evals_high.sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(list(components_range), errors, "o-", color=COLOURS["noise"],
         markersize=5, linewidth=2)
ax1.set_xlabel("Number of Components")
ax1.set_ylabel("Mean Reconstruction Error")
ax1.set_title("Error drops sharply at intrinsic dimension")

ax2.plot(list(components_range), explained, "o-", color=COLOURS["signal"],
         markersize=5, linewidth=2)
ax2.axhline(y=0.95, color=COLOURS["accent"], linestyle="--", alpha=0.7, label="95%")
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Variance Explained")
ax2.set_title("Variance explained jumps at intrinsic dimension")
ax2.legend()
ax2.set_ylim(0, 1.05)

fig.suptitle("50D data with 5 intrinsic dimensions", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03b_error_vs_components.png", dpi=150, bbox_inches="tight")
plt.show()

n_95 = np.searchsorted(explained, 0.95) + 1
print(f"True intrinsic dimension: {Z_true.shape[1]}")
print(f"Components for 95% variance: {n_95}")

## The Whole Thing in 20 Lines

Let's wrap our from-scratch PCA into a clean function.

In [ ]:
def pca_from_scratch(X, n_components):
    """PCA implemented from numpy only."""
    # Step 1: Centre
    mean = X.mean(axis=0)
    X_c = X - mean

    # Step 2: Covariance matrix
    C = np.cov(X_c.T)

    # Step 3: Eigendecomposition
    eigenvalues, eigenvectors = np.linalg.eigh(C)
    idx = np.argsort(eigenvalues)[::-1]
    eigenvalues = eigenvalues[idx]
    eigenvectors = eigenvectors[:, idx]

    # Step 4: Project
    W = eigenvectors[:, :n_components]
    X_pca = X_c @ W

    # Explained variance
    explained = eigenvalues[:n_components] / eigenvalues.sum()

    return X_pca, W, explained, mean


# Test on digits
data, target, images = make_digit_data()

X_ours, W_ours, explained_ours, mean_ours = pca_from_scratch(data, n_components=2)

pca_sk = PCA(n_components=2)
X_sk = pca_sk.fit_transform(data)

# Align signs (eigenvector sign is arbitrary)
for col in range(2):
    if np.corrcoef(X_ours[:, col], X_sk[:, col])[0, 1] < 0:
        X_ours[:, col] *= -1

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for digit in range(10):
    mask = target == digit
    ax1.scatter(X_ours[mask, 0], X_ours[mask, 1], s=8, alpha=0.5,
               label=str(digit), color=PALETTE[digit % len(PALETTE)])
    ax2.scatter(X_sk[mask, 0], X_sk[mask, 1], s=8, alpha=0.5,
               label=str(digit), color=PALETTE[digit % len(PALETTE)])

ax1.set_title("Our PCA (from scratch)")
ax1.set_xlabel("PC1")
ax1.set_ylabel("PC2")

ax2.set_title("sklearn PCA")
ax2.set_xlabel("PC1")
ax2.set_ylabel("PC2")
ax2.legend(title="Digit", markerscale=3, fontsize=8, ncol=2)

fig.suptitle("64D digits → 2D: our implementation matches sklearn", fontweight="bold")
plt.tight_layout()
plt.savefig("visuals/03b_scratch_vs_sklearn.png", dpi=150, bbox_inches="tight")
plt.show()

max_diff = np.max(np.abs(X_ours - X_sk))
print(f"Max difference: {max_diff:.2e}")
print(f"Explained variance (ours): {explained_ours.round(4)}")
print(f"Explained variance (sklearn): {pca_sk.explained_variance_ratio_.round(4)}")

<details>
<summary><b>The Maths Behind This</b></summary>

**The PCA algorithm in 5 equations:**

1. **Centre:** $\bar{X} = X - \mu$ where $\mu = \frac{1}{n} \sum_i \mathbf{x}_i$

2. **Covariance:** $C = \frac{1}{n-1} \bar{X}^T \bar{X}$

3. **Eigendecompose:** $C = Q \Lambda Q^T$ where $\Lambda = \text{diag}(\lambda_1, \ldots, \lambda_d)$ with $\lambda_1 \geq \ldots \geq \lambda_d$

4. **Project:** $Z = \bar{X} W$ where $W = Q_{:, 1:k}$ (first $k$ eigenvectors)

5. **Reconstruct:** $\hat{X} = Z W^T + \mu$

**Why `eigh` and not `eig`?** The covariance matrix is symmetric positive semi-definite. `np.linalg.eigh` exploits this symmetry for faster, more stable computation. Always use `eigh` for covariance matrices.

**Reconstruction error** for $k$ components:

$$E_k = \sum_{i=k+1}^{d} \lambda_i = \text{tr}(C) - \sum_{i=1}^{k} \lambda_i$$

The error equals the sum of the discarded eigenvalues.

</details>

## Where This Matters

**Understanding beats calling.** Anyone can call `PCA(n_components=5).fit_transform(X)`. But when results look wrong — when the first component captures suspiciously little variance, when the reconstruction is terrible — you need to understand the internals to debug. Building it from scratch gives you that understanding.

**Custom variants.** The from-scratch version is your starting point for weighted PCA (give some samples more importance), kernel PCA (nonlinear extension), or robust PCA (downweight outliers). These all modify specific steps — centering, covariance, or projection — and you can only modify what you understand.

## Feynman Check

1. **Why do we centre the data before computing covariance?**

2. **What do the eigenvectors of the covariance matrix represent?**

3. **You've just implemented PCA from scratch in ~20 lines. Verify it matches sklearn on a dataset of your choice.**

Now that you've built PCA by hand, let's apply it to real datasets and learn the practical decisions — how many components, when it works, when it fails — in **03c — PCA Applied**.